# 03 — Feature Engineering

Raw AspectTheme counts are a 405-column sparse matrix — most cells are 0 for any single call.  
This notebook condenses them into ~70 **interpretable, dense** features that a linear or tree model can actually learn from.

Feature groups we build:
1. **Magnitude-weighted net sentiment** per (Aspect × Theme) — 45 features
2. **Speaker disagreement** — CEO vs Analyst, Presentation vs Answer tone divergence
3. **Kitchen-sink pattern** — bad current results + positive forward guidance (historically bullish)
4. **QoQ trend and acceleration** — rolling quarter-over-quarter delta per ticker
5. **Sector-relative percentile rank** of ATC score — expanding window, point-in-time
6. **Transcript length** control feature

**Look-ahead audit:** QoQ and percentile features use only events that have already occurred before the current event date. No peeking at the current quarter's cross-section.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

PROJECT      = Path(os.getenv("ATC_PROJECT_ROOT",
                              Path.cwd().parent if Path.cwd().name == 'notebooks'
                              else Path.cwd()))
DATA_DIR     = PROJECT / 'data'
SIGNALS_PQ   = DATA_DIR / 'signals.parquet'
AUGMENTED_PQ = DATA_DIR / 'signals_with_returns.parquet'
FEATURES_PQ  = DATA_DIR / 'features.parquet'

with open(DATA_DIR / 'col_registry.json') as f:
    col_reg = json.load(f)
ASPECT_COLS  = col_reg['aspect_cols']
PLACEBO_COLS = col_reg['placebo_cols']

# Load full signal (all SignalTypes) for multi-slice features
df_all = pd.read_parquet(SIGNALS_PQ)
# Load the augmented Total slice (has entry_date, universe flags, forward returns)
df_aug = pd.read_parquet(AUGMENTED_PQ)

print(f'Full signal  : {df_all.shape}')
print(f'Augmented    : {df_aug.shape}')
print(f'AspectTheme cols to use: {len(ASPECT_COLS)}')
print(f'Placebo cols (Fluff+Filler): {len(PLACEBO_COLS)}')
# Verify placebo cols are actually in the parquet now
present = sum(1 for c in PLACEBO_COLS if c in df_all.columns)
print(f'Placebo cols present in signals.parquet: {present}/{len(PLACEBO_COLS)}')


Full signal  : (2738206, 600)
Augmented    : (376790, 609)
AspectTheme cols to use: 405
Placebo cols (Fluff+Filler): 162
Placebo cols present in signals.parquet: 162/162


## 3.1 Magnitude-Weighted Net Sentiment per (Aspect × Theme)

Raw AspectTheme columns are sentence **counts** per bucket: `AspectTheme_{Aspect}_{Theme} - {High/Medium/Low} - {Positive/Neutral/Negative}`.

We collapse each (Aspect × Theme) pair into a single weighted net-sentiment score:
```
net = (High_Pos×3 + Med_Pos×2 + Low_Pos×1) - (High_Neg×3 + Med_Neg×2 + Low_Neg×1)
norm_net = net / max(total_sentences, 1)
```
This preserves magnitude information (High-confidence sentences count more) while reducing 405 sparse columns to **45 dense features** (5 useful Aspects × 9 Themes).

In [2]:
USEFUL_ASPECTS = ['CurrentState','Forecast','Surprise','StrategicPosition','Other']
THEMES = [
    'FinancialPerformance','OperationalPerformance','MarketAndCompetitivePosition',
    'StrategicInitiatives','CapitalAllocation','RegulatoryAndLegalIssues',
    'ESG','MacroeconomicFactors','Other'
]
MAGNITUDES  = ['High','Medium','Low']
MAG_WEIGHTS = {'High': 3, 'Medium': 2, 'Low': 1}

# Build a lookup: (aspect, theme, magnitude, sentiment) → column name
def col_name(aspect, theme, magnitude, sentiment):
    return f'AspectTheme_{aspect}_{theme} - {magnitude} - {sentiment}'

total_slice = df_all[df_all['SignalType'] == 'Total'].copy()

feat_rows = []
for aspect in USEFUL_ASPECTS:
    for theme in THEMES:
        pos_num = np.zeros(len(total_slice))
        neg_num = np.zeros(len(total_slice))
        for mag in MAGNITUDES:
            w = MAG_WEIGHTS[mag]
            c_pos = col_name(aspect, theme, mag, 'Positive')
            c_neg = col_name(aspect, theme, mag, 'Negative')
            if c_pos in total_slice.columns:
                pos_num += w * total_slice[c_pos].fillna(0).values
            if c_neg in total_slice.columns:
                neg_num += w * total_slice[c_neg].fillna(0).values
        net      = pos_num - neg_num
        denom    = total_slice['Sentences'].fillna(1).clip(lower=1).values
        feat_name = f'mwns_{aspect}_{theme}'
        total_slice[feat_name] = net / denom

mwns_cols = [c for c in total_slice.columns if c.startswith('mwns_')]
print(f'Magnitude-weighted net sentiment features: {len(mwns_cols)}')
print(total_slice[mwns_cols].describe().round(4).T[['mean','std','min','max']].head(10))

Magnitude-weighted net sentiment features: 45


                                                  mean     std      min  \
mwns_CurrentState_FinancialPerformance          0.2165  0.2649 -20.5000   
mwns_CurrentState_OperationalPerformance        0.1762  0.1510  -2.0000   
mwns_CurrentState_MarketAndCompetitivePosition  0.0535  0.0711  -2.5000   
mwns_CurrentState_StrategicInitiatives          0.0401  0.0487  -0.6667   
mwns_CurrentState_CapitalAllocation             0.0170  0.0291  -0.3333   
mwns_CurrentState_RegulatoryAndLegalIssues      0.0054  0.0259  -1.0000   
mwns_CurrentState_ESG                           0.0052  0.0161  -1.2500   
mwns_CurrentState_MacroeconomicFactors         -0.0142  0.0399  -2.6667   
mwns_CurrentState_Other                        -0.0013  0.0137  -0.6667   
mwns_Forecast_FinancialPerformance              0.0965  0.0974  -3.5000   

                                                    max  
mwns_CurrentState_FinancialPerformance          24.2500  
mwns_CurrentState_OperationalPerformance        13.5000  


## 3.2 Speaker Disagreement Features

Each earnings call has multiple SignalType slices. Comparing them reveals *who* is responsible for positive/negative tone:
- **CEO vs Analyst disagreement**: when CEOs are notably more optimistic than the questions analysts ask, management may be overselling — bearish signal
- **Answer vs Presentation delta**: management's scripted remarks vs their unscripted Q&A answers. A negative delta (management goes more negative in answers) is often predictive of post-earnings drift

In [3]:
# Pivot ATC score by SignalType for each DocID
pivot = (
    df_all
    .loc[df_all['SignalType'].isin(['Total','CEO','CFO','Analysts','Presentation','Answer','Question']),
         ['DocID','SignalType','ATCClassifierScore']]
    .pivot_table(index='DocID', columns='SignalType', values='ATCClassifierScore')
)
pivot.columns = [f'atc_{c.lower()}' for c in pivot.columns]
pivot = pivot.reset_index()

# Divergence features
if 'atc_ceo' in pivot.columns and 'atc_analysts' in pivot.columns:
    pivot['spk_ceo_minus_analyst']  = pivot['atc_ceo']  - pivot['atc_analysts']
if 'atc_ceo' in pivot.columns and 'atc_cfo' in pivot.columns:
    pivot['spk_ceo_minus_cfo']      = pivot['atc_ceo']  - pivot['atc_cfo']
if 'atc_answer' in pivot.columns and 'atc_presentation' in pivot.columns:
    pivot['spk_answer_minus_pres']  = pivot['atc_answer'] - pivot['atc_presentation']
if 'atc_question' in pivot.columns and 'atc_answer' in pivot.columns:
    pivot['spk_mgmt_response_bias'] = pivot['atc_answer'] - pivot['atc_question']

spk_cols = [c for c in pivot.columns if c.startswith('spk_') or c.startswith('atc_')]
print(f'Speaker pivot columns: {spk_cols}')

# Merge back to total slice
total_slice = total_slice.merge(pivot, on='DocID', how='left')
print(f'After merge: {total_slice.shape}')

Speaker pivot columns: ['atc_analysts', 'atc_answer', 'atc_ceo', 'atc_cfo', 'atc_presentation', 'atc_question', 'atc_total', 'spk_ceo_minus_analyst', 'spk_ceo_minus_cfo', 'spk_answer_minus_pres', 'spk_mgmt_response_bias']


After merge: (376790, 656)


## 3.3 Kitchen-Sink Pattern

*"Bad current, good forward"* — a company reports poor current-period results but provides positive forward guidance. This "kitchen-sink quarter" pattern is historically bullish: the stock has already been punished, and forward guidance resets expectations.

We operationalise it as the interaction between `CurrentState_FinancialPerformance` (should be negative) and `Forecast_FinancialPerformance` (should be positive).

In [4]:
# mwns_CurrentState_FinancialPerformance is negative for bad current results
# mwns_Forecast_FinancialPerformance is positive for optimistic guidance
cs_fp  = total_slice.get('mwns_CurrentState_FinancialPerformance', pd.Series(0.0, index=total_slice.index))
fc_fp  = total_slice.get('mwns_Forecast_FinancialPerformance',     pd.Series(0.0, index=total_slice.index))

# Kitchen-sink score: highest when current is negative AND forecast is positive
total_slice['ks_bad_now_good_fwd']  = (-cs_fp).clip(lower=0) * fc_fp.clip(lower=0)

# Reverse: "sandbagging" — positive current but negative guidance
total_slice['ks_good_now_bad_fwd']  = cs_fp.clip(lower=0) * (-fc_fp).clip(lower=0)

# Surprise × Forecast interaction: unexpected surprise + positive guidance = strong signal
surp = total_slice.get('mwns_Surprise_FinancialPerformance', pd.Series(0.0, index=total_slice.index))
total_slice['ks_surprise_fwd']      = surp * fc_fp

print('Kitchen-sink features:')
print(total_slice[['ks_bad_now_good_fwd','ks_good_now_bad_fwd','ks_surprise_fwd']].describe().round(4).T)

Kitchen-sink features:
                        count    mean     std   min  25%  50%     75%     max
ks_bad_now_good_fwd  376790.0  0.0012  0.0837 -0.00  0.0  0.0  0.0000  45.000
ks_good_now_bad_fwd  376790.0  0.0003  0.0220 -0.00  0.0  0.0  0.0000   7.840
ks_surprise_fwd      376790.0  0.0005  0.0173 -1.28  0.0  0.0  0.0002   3.875


## 3.4 QoQ Trend and Acceleration

We compute rolling quarter-over-quarter deltas of `ATCClassifierScore` per ticker:
- **QoQ delta**: current ATC − mean of prior 4 quarters → momentum / trend-reversal signal
- **QoQ acceleration**: current ATC − 2×(1Q ago) + (2Q ago) → second derivative of sentiment

**Look-ahead safety**: we sort by (ticker, entry_date) and use `.shift()` — the current row never sees its own value when computing the lag.

In [5]:
# Sort so shift() is strictly backward-looking
total_slice = total_slice.sort_values(['BESTTICKER','DocDate']).reset_index(drop=True)

grp = total_slice.groupby('BESTTICKER')['ATCClassifierScore']

# Prior-4-quarter mean (shifted so we don't include current call)
total_slice['qoq_prior4_mean'] = grp.transform(
    lambda s: s.shift(1).rolling(4, min_periods=1).mean()
)
total_slice['qoq_delta']       = total_slice['ATCClassifierScore'] - total_slice['qoq_prior4_mean']

# Acceleration: ATC_t - 2*ATC_{t-1} + ATC_{t-2}
lag1 = grp.transform(lambda s: s.shift(1))
lag2 = grp.transform(lambda s: s.shift(2))
total_slice['qoq_acceleration'] = total_slice['ATCClassifierScore'] - 2*lag1 + lag2

# Trend reversal flag: was negative for 2+ quarters, now positive?
neg_2q = (lag1 < 0) & (lag2 < 0)
total_slice['qoq_reversal_up']   = ((total_slice['ATCClassifierScore'] > 0) & neg_2q).astype(float)

qoq_cols = ['qoq_delta','qoq_acceleration','qoq_reversal_up']
print(total_slice[qoq_cols].describe().round(4).T)

                     count    mean     std     min     25%     50%     75%  \
qoq_delta         359125.0  0.0002  0.0334 -0.2833 -0.0182  0.0006  0.0189   
qoq_acceleration  342744.0 -0.0001  0.0624 -0.5524 -0.0355 -0.0008  0.0347   
qoq_reversal_up   376790.0  0.0538  0.2257  0.0000  0.0000  0.0000  0.0000   

                     max  
qoq_delta         0.2373  
qoq_acceleration  0.5370  
qoq_reversal_up   1.0000  


## 3.5 Sector-Relative Percentile Rank (Expanding Window)

Tech earnings calls are structurally more upbeat than Utilities. Raw ATC scores conflate sector tone with stock-specific signal.

We compute each call's **expanding-window percentile rank within its GICS sector** — what fraction of all prior calls in the same sector had a lower ATC score than this one.

**Look-ahead safety**: `expanding()` with `min_periods=10` only uses events that have occurred *before the current event* — no cross-sectional leakage.

In [6]:
def expanding_pct_rank(series: pd.Series) -> pd.Series:
    """
    Percentile rank of each element vs ALL PRIOR elements (strictly backward-looking).
    Sorted by entry_date ensures same-day calls see only earlier trade-days.
    Requires min 10 prior observations before returning a value.
    """
    result = pd.Series(np.nan, index=series.index)
    for i in range(1, len(series)):
        hist = series.iloc[:i].dropna()
        if len(hist) < 10:
            continue
        val = series.iloc[i]
        if pd.notna(val):
            result.iloc[i] = (hist < val).mean()
    return result

# Sort by SECTOR then entry_date (trade date, not call date) to avoid
# same-day call-time leakage across companies reporting on the same calendar day
total_slice = total_slice.merge(
    df_aug[['DocID', 'entry_date']], on='DocID', how='left'
)
total_slice = total_slice.sort_values(['SECTOR', 'entry_date']).reset_index(drop=True)

total_slice['sector_pct_rank'] = (
    total_slice
    .groupby('SECTOR')['ATCClassifierScore']
    .transform(expanding_pct_rank)
)

# Re-sort by entry_date for downstream use
total_slice = total_slice.sort_values('entry_date').reset_index(drop=True)

print('Sector percentile rank coverage:')
print(f"  Valid: {total_slice['sector_pct_rank'].notna().sum():,} / {len(total_slice):,}")
print(total_slice['sector_pct_rank'].describe().round(3).to_string())


Sector percentile rank coverage:
  Valid: 376,680 / 376,790
count    376680.000
mean          0.510
std           0.292
min           0.000
25%           0.257
50%           0.515
75%           0.765
max           1.000


## 3.6 Transcript Length Control

Longer transcripts produce more raw sentence counts in every AspectTheme bucket. We add a length-normalized control feature so the model doesn't learn to just like verbose calls.

In [7]:
total_slice['log_sentences'] = np.log1p(total_slice['Sentences'].fillna(0))

# Sector-relative sentence length z-score.
# Use shift(1) before expanding so the current call is NOT included in its own mean/std.
# This is fully point-in-time: we only normalise against prior calls in the same sector.
def prior_expanding_z(s):
    shifted = s.shift(1)
    mu = shifted.expanding(min_periods=5).mean()
    sd = shifted.expanding(min_periods=5).std().clip(lower=1e-6)
    return (s - mu) / sd

total_slice['sentences_sector_z'] = (
    total_slice.groupby('SECTOR')['log_sentences']
    .transform(prior_expanding_z)
)
print('Length control features added (sentences_sector_z uses shift(1) — fully PIT).')


Length control features added (sentences_sector_z uses shift(1) — fully PIT).


## 3.7 Assemble Final Feature Matrix and Save

Collect all engineered feature columns into one dataframe alongside the join keys (`DocID`, `BESTTICKER`, `entry_date`), universe flags, and forward return targets.

In [8]:
# Merge engineered features into the augmented df (which has fwd returns + universe flags)
all_feat_cols = (
    mwns_cols
    + [c for c in pivot.columns if c.startswith('atc_') or c.startswith('spk_')]
    + ['ks_bad_now_good_fwd','ks_good_now_bad_fwd','ks_surprise_fwd']
    + qoq_cols
    + ['sector_pct_rank','log_sentences','sentences_sector_z']
    + ['ATCClassifierScore']     # keep raw ATC as a feature too
    + col_reg['event_cols']      # EventScore variants
)

join_cols     = ['DocID','BESTTICKER','DocDate','entry_date','SECTOR',
                 'in_sp500','in_sp1500','in_ru3k','QTR_YEAR']
target_cols   = [f'fwd_{h}d' for h in [1,3,5,10,20]]

# Merge engineered cols into total_slice (which already has most things)
# total_slice was built from the signals parquet, df_aug has universe flags & returns
feat_df = total_slice.merge(
    df_aug[['DocID'] + ['in_sp500','in_sp1500','in_ru3k','entry_date'] + target_cols],
    on='DocID', how='left', suffixes=('','_aug')
)

keep = [c for c in join_cols + all_feat_cols + target_cols if c in feat_df.columns]
feat_df = feat_df[keep].drop_duplicates(subset=['DocID'])

feat_df.to_parquet(FEATURES_PQ, index=False)
print(f'Feature matrix saved → {FEATURES_PQ}')
print(f'Shape: {feat_df.shape}')
print(f'\nFeature groups:')
print(f'  MWNS (Aspect×Theme)   : {len(mwns_cols)}')
print(f'  Speaker divergence    : {len([c for c in feat_df.columns if c.startswith("spk_")])}')
print(f'  ATC per SignalType    : {len([c for c in feat_df.columns if c.startswith("atc_")])}')
print(f'  Kitchen-sink patterns : 3')
print(f'  QoQ trend             : {len(qoq_cols)}')
print(f'  Sector rank / length  : 3')
print(f'  EventScores           : {len(col_reg["event_cols"])}')
print(f'  ATCClassifierScore    : 1')
print(f'\nNull pct in feature cols:')
feat_nulls = feat_df[all_feat_cols].isnull().mean().sort_values(ascending=False)
print(feat_nulls[feat_nulls > 0.05].head(10).round(3).to_string())

Feature matrix saved → /Users/chaithanyapakala/Downloads/NLP_FinalProject/data/features.parquet
Shape: (376790, 92)

Feature groups:
  MWNS (Aspect×Theme)   : 45
  Speaker divergence    : 4
  ATC per SignalType    : 7
  Kitchen-sink patterns : 3
  QoQ trend             : 3
  Sector rank / length  : 3
  EventScores           : 12
  ATCClassifierScore    : 1

Null pct in feature cols:
spk_ceo_minus_cfo         0.434
atc_cfo                   0.328
spk_ceo_minus_analyst     0.259
atc_ceo                   0.194
qoq_acceleration          0.090
atc_analysts              0.088
spk_mgmt_response_bias    0.071
atc_question              0.067
spk_answer_minus_pres     0.054


## 3.8 Placebo Feature Matrix (Fluff + Filler only)

Build a **placebo** feature matrix using only the Fluff/Filler AspectTheme columns. Per the handout, a signal built entirely on these noise classes should produce ≈0 alpha. If the notebook 04 placebo backtest shows material Sharpe, there is a systematic look-ahead bug somewhere.

This is our red-team sanity check and a compelling result to include in the research PDF.

In [9]:
placebo_slice = df_all[df_all['SignalType'] == 'Total'].copy()

# Build a simple placebo ATC: sum of all Fluff/Filler positive minus negative counts
plac_pos = [c for c in PLACEBO_COLS if 'Positive' in c]
plac_neg = [c for c in PLACEBO_COLS if 'Negative' in c]

# Null-safe sum
def safe_col(df, cols):
    existing = [c for c in cols if c in df.columns]
    return df[existing].fillna(0).sum(axis=1) if existing else pd.Series(0, index=df.index)

placebo_slice['placebo_signal'] = (
    (safe_col(placebo_slice, plac_pos) - safe_col(placebo_slice, plac_neg))
    / placebo_slice['Sentences'].fillna(1).clip(lower=1)
)

placebo_df = placebo_slice[['DocID','BESTTICKER','DocDate','placebo_signal']].copy()
placebo_df = placebo_df.merge(
    df_aug[['DocID','entry_date','in_sp500','in_sp1500','in_ru3k'] + target_cols],
    on='DocID', how='left'
)
placebo_df.to_parquet(DATA_DIR / 'placebo_features.parquet', index=False)
print(f'Placebo feature matrix saved. Shape: {placebo_df.shape}')
print(placebo_df['placebo_signal'].describe().round(4).to_string())

Placebo feature matrix saved. Shape: (376790, 13)
count    376790.0000
mean          0.0276
std           0.0206
min          -0.2500
25%           0.0140
50%           0.0240
75%           0.0370
max           1.0000
